In [ ]:
!pip install transformers datasets

In [ ]:
from datasets import load_dataset

# Pulls the exact same dataset, but from a modernized, Parquet-supported repo
dataset = load_dataset("thesofakillers/jigsaw-toxic-comment-classification-challenge")

# Tip: Print the dataset to see its structure!
print(dataset)

In [ ]:
!pip install datasets==2.16.0 huggingface-hub==0.20.0

Model Initialization and training

In [ ]:
!pip install --upgrade transformers datasets huggingface_hub

In [5]:
from transformers import AutoModelForSequenceClassification

# Load the base model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=6,
    problem_type="multi_label_classification"
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
#Phase 3: Import training auguements
from transformers import TrainingArguments

In [13]:
# Define hyperparameters
# Define how the model will learn
training_args = TrainingArguments(
    output_dir="./toxicity-guardian-model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",  # <-- This was changed to match the latest version
    save_strategy="epoch"
)

print("Training arguments locked in!")

Training arguments locked in!


In [14]:
# Import Trainer API
from transformers import Trainer

In [17]:
from datasets import load_dataset
from transformers import AutoTokenizer

# 1. Load the modernized dataset
dataset = load_dataset("thesofakillers/jigsaw-toxic-comment-classification-challenge")

# 2. Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# 3. Tokenize with a safety check for empty/null rows
def tokenize_function(examples):
    # This converts everything to a string and handles any None/NaN values
    safe_texts = [str(text) if text is not None else "" for text in examples["comment_text"]]
    return tokenizer(safe_texts, padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

print("Data is successfully reloaded and tokenized!")

Map:   0%|          | 0/159571 [00:00<?, ? examples/s]

Map:   0%|          | 0/306328 [00:00<?, ? examples/s]

Data is successfully reloaded and tokenized!


In [30]:
# Create a list of the 6 specific toxicity label columns
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# Define a function to bundle them into a single list of floats with a safety check
def bundle_labels(examples):
    batch_labels = []
    # Loop through the batch and group the 6 labels together for each comment
    for i in range(len(examples["toxic"])):
        # Safety check: if the label is missing (None), treat it as 0.0
        safe_labels = [float(examples[col][i]) if examples[col][i] is not None else 0.0 for col in label_cols]
        batch_labels.append(safe_labels)
    return {"labels": batch_labels}

print("Bundling the 6 toxicity columns into a single 'labels' format...")

# Apply the formatting to the dataset
tokenized_datasets = tokenized_datasets.map(bundle_labels, batched=True)

print("Labels successfully bundled! The data is fully ready.")

Bundling the 6 toxicity columns into a single 'labels' format...


Map:   0%|          | 0/159571 [00:00<?, ? examples/s]

Map:   0%|          | 0/306328 [00:00<?, ? examples/s]

Labels successfully bundled! The data is fully ready.


In [31]:
# Initialize the Trainer (Run this again!)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
)

print("Trainer API is set up and ready to go!")

Trainer API is set up and ready to go!


In [32]:
print("Starting the training process... grab a coffee! ☕")

# Execute the training loop
trainer.train()

print("Training complete! Your model is officially fine-tuned.")

Starting the training process... grab a coffee! ☕


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,0.035955,-1.254450


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [33]:
import os

# Create a folder for VibeGuard
save_directory = "./VibeGuard-final"

print(f"Saving your fine-tuned model to {save_directory}...")

# Save the model weights and the tokenizer
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"✅ Success! VibeGuard is now safely saved in the folder: {save_directory}")

Saving your fine-tuned model to ./VibeGuard-final...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Success! VibeGuard is now safely saved in the folder: ./VibeGuard-final


In [34]:
# Install streamlit and localtunnel
!pip install -q streamlit
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 49.2 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

In [40]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report

print("Loading VibeGuard model...")
classifier = pipeline("text-classification", model="./VibeGuard-final", tokenizer="./VibeGuard-final", top_k=None)

# The "Golden Test Set"
# 0 = Safe (Good Vibes), 1 = Toxic (Bad Vibes)
golden_test_set = [
    {"text": "You are an amazing teammate, thanks for the help!", "toxic": 0},
    {"text": "I strongly disagree with your approach, it's highly inefficient.", "toxic": 0}, # Critical but safe
    {"text": "Could you please explain why you made this change?", "toxic": 0},
    {"text": "I think we should rewrite this entire section.", "toxic": 0},
    {"text": "You are a complete idiot and I hate you.", "toxic": 1},
    {"text": "If you don't delete this post, I will find where you live.", "toxic": 1}, # Threat
    {"text": "Shut up you dumb piece of trash.", "toxic": 1},
    {"text": "Nobody likes you and you should quit.", "toxic": 1},
    {"text": "I appreciate your perspective, but I think it's wrong.", "toxic": 0},
    {"text": "Go to hell you stupid moron.", "toxic": 1},
    {"text": "Wow, this is a really fascinating article.", "toxic": 0},
    {"text": "I will literally break your legs.", "toxic": 1}, # Severe threat
    {"text": "This UI looks terrible, you guys suck at design.", "toxic": 1},
    {"text": "Let's grab coffee tomorrow and discuss the architecture.", "toxic": 0},
    {"text": "You are absolutely pathetic and worthless.", "toxic": 1}
]

true_labels = []
predicted_labels = []

print("Evaluating model on Golden Test Set...")
for item in golden_test_set:
    text = item["text"]
    true_labels.append(item["toxic"])

    # Model prediction
    results = classifier(text, truncation=True, max_length=512)[0]

    # 🔥 THE FIX: If ANY of the 6 labels score higher than 50%, flag it as toxic!
    # This works perfectly whether the labels are named 'toxic' or 'LABEL_0'
    is_predicted_toxic = 1 if any(res['score'] > 0.5 for res in results) else 0
    predicted_labels.append(is_predicted_toxic)

# Calculate final metrics
accuracy = accuracy_score(true_labels, predicted_labels)
print("\n=== ✨ VibeGuard Evaluation Metrics ✨ ===")
print(f"Overall Accuracy: {accuracy * 100:.2f}%\n")
print("Detailed Report:")
print(classification_report(true_labels, predicted_labels, target_names=["Safe", "Toxic"]))

Loading VibeGuard model...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Evaluating model on Golden Test Set...

=== ✨ VibeGuard Evaluation Metrics ✨ ===
Overall Accuracy: 86.67%

Detailed Report:
              precision    recall  f1-score   support

        Safe       0.78      1.00      0.88         7
       Toxic       1.00      0.75      0.86         8

    accuracy                           0.87        15
   macro avg       0.89      0.88      0.87        15
weighted avg       0.90      0.87      0.87        15



In [1]:
%%writefile app.py
import streamlit as st
from transformers import pipeline

st.set_page_config(page_title="VibeGuard ✨", page_icon="✨", layout="centered")

st.markdown("""
<style>
    .stApp { background-color: #FAFAFA; font-family: 'Inter', sans-serif; }
    .title-wrapper { text-align: center; padding-bottom: 2rem; }
    .stTextArea textarea { border-radius: 15px !important; border: 2px solid #EAEAEA !important; padding: 15px; font-size: 16px; }
    .stTextArea textarea:focus { border-color: #6C63FF !important; }
    .stButton button { border-radius: 25px !important; background-color: #6C63FF !important; color: white !important; width: 100%; margin-top: 10px; }
    .result-card { background: white; padding: 25px; border-radius: 20px; box-shadow: 0px 10px 30px rgba(0,0,0,0.05); margin-top: 20px; }
    .vibe-pass { background-color: #E8F5E9; color: #2E7D32; padding: 15px; border-radius: 10px; text-align: center; font-weight: bold; }
    .vibe-fail { background-color: #FFEBEE; color: #C62828; padding: 15px; border-radius: 10px; text-align: center; font-weight: bold; }
</style>
""", unsafe_allow_html=True)

st.markdown("<div class='title-wrapper'><h1>✨ VibeGuard</h1><p style='color: #666;'>Your AI Trust & Safety Guardian.</p></div>", unsafe_allow_html=True)

@st.cache_resource
def load_model():
    return pipeline("text-classification", model="./VibeGuard-final", tokenizer="./VibeGuard-final", top_k=None)

classifier = load_model()
user_input = st.text_area("What's on your mind?", height=120, placeholder="Type a comment to check its vibe...")

if st.button("Check the Vibe ✨"):
    if user_input.strip() != "":
        with st.spinner("Sensing the vibes..."):
            results = classifier(user_input)[0]
            results = sorted(results, key=lambda x: x['score'], reverse=True)
            is_toxic = any(result['score'] > 0.5 for result in results)

            st.markdown("<div class='result-card'>", unsafe_allow_html=True)
            if is_toxic:
                st.markdown("<div class='vibe-fail'>🛑 Vibe Check Failed: Toxicity Detected</div>", unsafe_allow_html=True)
            else:
                st.markdown("<div class='vibe-pass'>🌿 Passed the Vibe Check! Good vibes only.</div>", unsafe_allow_html=True)

            st.markdown("### Deep Dive Metrics")
            for result in results:
                col1, col2 = st.columns([1, 4])
                with col1: st.write(f"**{result['label'].replace('_', ' ').title()}**")
                with col2: st.progress(result['score'])
            st.markdown("</div>", unsafe_allow_html=True)

Writing app.py
